# 04 — Adversarial Evaluation

Compare False Acceptance Rate under human impostors vs VAE / GAN synthetic impersonation.

Prefer regenerating results with:

```bash
python scripts/run_experiment.py --subjects 10 --classifiers svm,random_forest --no-tune
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_dataset, list_subjects
from src.data.splits import build_subject_split, build_training_dataset, build_verification_dataset
from src.models.classifier import train_policy_engine, predict_proba
from src.adversarial.synthesize import generate_impersonation_samples
from src.evaluation.metrics import compute_metrics, compute_attack_metrics, summarize_attack_results

df = load_dataset()
subjects = list_subjects(df)[:3]
attack_rows = []
for subject in subjects:
    split = build_subject_split(df, subject, 'hold_flight')
    Xtr, ytr = build_training_dataset(split, impostor_samples_per_user=640)
    Xv, yv = build_verification_dataset(split)
    eng = train_policy_engine(Xtr, ytr, subject=subject, feature_set='hold_flight',
                              classifier_type='svm', tune_hyperparameters=False)
    scores = predict_proba(eng, Xv)
    m = compute_metrics(yv, scores, subject=subject)
    n_g = len(split.X_verify)
    for method in ('vae', 'gan'):
        epochs = 80 if method == 'vae' else 120
        samples, _ = generate_impersonation_samples(
            split.X_enroll, subject=subject, feature_set='hold_flight',
            method=method, n_samples=80, train_kwargs={'epochs': epochs},
        )
        ss = predict_proba(eng, samples)
        atk = compute_attack_metrics(
            subject=subject, method=method, classifier_type='svm',
            genuine_scores=scores[:n_g], human_impostor_scores=scores[n_g:],
            synthetic_scores=ss, threshold=m.threshold,
        )
        attack_rows.append(atk)
        print(subject, method, f"FAR_h={atk.far_human:.3f} FAR_a={atk.far_attack:.3f}")

summary = summarize_attack_results(attack_rows)
summary

In [ ]:
detail = pd.DataFrame([r.__dict__ for r in attack_rows])
plot_df = detail.melt(id_vars=['subject','method'], value_vars=['far_human','far_attack'],
                      var_name='type', value_name='far')
plt.figure(figsize=(8,4))
sns.barplot(data=plot_df, x='method', y='far', hue='type')
plt.ylim(0,1)
plt.title('FAR: Human impostors vs generative impersonation')
plt.tight_layout()
plt.show()